# Celebal Week 3 Assignment

**Author: Rakshit Gupta**


# Superstore Sales Analysis using SQL

## 1. Setup & Dependencies <a id='setup'></a>

In [3]:
import pandas as pd
import sqlite3
import warnings
warnings.filterwarnings('ignore')

# Pretty display settings
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 120)
pd.set_option('display.float_format', '{:.2f}'.format)

print("✅ Libraries loaded successfully!")


✅ Libraries loaded successfully!


## 2. Load Dataset into SQLite <a id='load'></a>


In [7]:
# Load CSV into pandas
df = pd.read_csv(r'C:\Users\HP\Desktop\Sample - Superstore.csv ', encoding='windows-1252')

print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
df.head(3)


Shape: (9994, 21)

Columns: ['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State', 'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category', 'Product Name', 'Sales', 'Quantity', 'Discount', 'Profit']


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.96,2,0.00,41.91
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.94,3,0.00,219.58
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.62,2,0.00,6.87


In [8]:
# Create SQLite in-memory database and load raw data
conn = sqlite3.connect(':memory:')

# Load entire dataframe as superstore_raw
df.to_sql('superstore_raw', conn, if_exists='replace', index=False)

# Verify
result = pd.read_sql("SELECT COUNT(*) AS total_rows FROM superstore_raw", conn)
print("✅ superstore_raw loaded with", result['total_rows'][0], "rows")


✅ superstore_raw loaded with 9994 rows


In [9]:
# Preview raw table
pd.read_sql("""
    SELECT [Row ID], [Order ID], [Order Date], [Customer ID], [Customer Name],
           [Segment], [Category], [Sub-Category], [Product Name], Sales, Quantity, Discount, Profit
    FROM superstore_raw
    LIMIT 5
""", conn)


,Row ID,Order ID,Order Date,Customer ID,Customer Name,Segment,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11/8/2016,CG-12520,Claire Gute,Consumer,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.96,2,0.00,41.91
1,2,CA-2016-152156,11/8/2016,CG-12520,Claire Gute,Consumer,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.94,3,0.00,219.58
2,3,CA-2016-138688,6/12/2016,DV-13045,Darrin Van Huff,Corporate,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.62,2,0.00,6.87
3,4,US-2015-108966,10/11/2015,SO-20335,Sean O'Donnell,Consumer,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.58,5,0.45,-383.03
4,5,US-2015-108966,10/11/2015,SO-20335,Sean O'Donnell,Consumer,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.37,2,0.20,2.52


## 3. Create Normalized Tables <a id='create'></a>

We extract **customers**, **products**, and **orders** tables from the raw data using `SELECT DISTINCT`.


In [10]:
# ── CUSTOMERS TABLE ──
conn.execute("""
    CREATE TABLE IF NOT EXISTS customers AS
    SELECT DISTINCT
        [Customer ID]   AS customer_id,
        [Customer Name] AS customer_name,
        [Segment]       AS segment,
        [City]          AS city,
        [State]         AS state,
        [Region]        AS region
    FROM superstore_raw
""")

count = pd.read_sql("SELECT COUNT(*) AS cnt FROM customers", conn)
print("✅ customers table created —", count['cnt'][0], "records")
pd.read_sql("SELECT * FROM customers LIMIT 5", conn)


✅ customers table created — 4688 records


,customer_id,customer_name,segment,city,state,region
0,CG-12520,Claire Gute,Consumer,Henderson,Kentucky,South
1,DV-13045,Darrin Van Huff,Corporate,Los Angeles,California,West
2,SO-20335,Sean O'Donnell,Consumer,Fort Lauderdale,Florida,South
3,BH-11710,Brosina Hoffman,Consumer,Los Angeles,California,West
4,AA-10480,Andrew Allen,Consumer,Concord,North Carolina,South


In [11]:
# ── PRODUCTS TABLE ──
conn.execute("""
    CREATE TABLE IF NOT EXISTS products AS
    SELECT DISTINCT
        [Product ID]    AS product_id,
        [Product Name]  AS product_name,
        [Category]      AS category,
        [Sub-Category]  AS sub_category
    FROM superstore_raw
""")

count = pd.read_sql("SELECT COUNT(*) AS cnt FROM products", conn)
print("✅ products table created —", count['cnt'][0], "records")
pd.read_sql("SELECT * FROM products LIMIT 5", conn)


✅ products table created — 1894 records


,product_id,product_name,category,sub_category
0,FUR-BO-10001798,Bush Somerset Collection Bookcase,Furniture,Bookcases
1,FUR-CH-10000454,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",Furniture,Chairs
2,OFF-LA-10000240,Self-Adhesive Address Labels for Typewriters b...,Office Supplies,Labels
3,FUR-TA-10000577,Bretford CR4500 Series Slim Rectangular Table,Furniture,Tables
4,OFF-ST-10000760,Eldon Fold 'N Roll Cart System,Office Supplies,Storage


In [12]:
# ── ORDERS TABLE ──
conn.execute("""
    CREATE TABLE IF NOT EXISTS orders AS
    SELECT DISTINCT
        [Row ID]        AS row_id,
        [Order ID]      AS order_id,
        [Order Date]    AS order_date,
        [Ship Date]     AS ship_date,
        [Ship Mode]     AS ship_mode,
        [Customer ID]   AS customer_id,
        [Product ID]    AS product_id,
        CAST(Sales      AS REAL) AS sales,
        CAST(Quantity   AS INTEGER) AS quantity,
        CAST(Discount   AS REAL) AS discount,
        CAST(Profit     AS REAL) AS profit
    FROM superstore_raw
""")

count = pd.read_sql("SELECT COUNT(*) AS cnt FROM orders", conn)
print("✅ orders table created —", count['cnt'][0], "records")
pd.read_sql("SELECT * FROM orders LIMIT 5", conn)


✅ orders table created — 9994 records


,row_id,order_id,order_date,ship_date,ship_mode,customer_id,product_id,sales,quantity,discount,profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,FUR-BO-10001798,261.96,2,0.00,41.91
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,FUR-CH-10000454,731.94,3,0.00,219.58
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,OFF-LA-10000240,14.62,2,0.00,6.87
3,4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,FUR-TA-10000577,957.58,5,0.45,-383.03
4,5,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,OFF-ST-10000760,22.37,2,0.20,2.52


## 4. Subqueries | CTE | Window Function <a id='subqueries'></a>

### 4.1 Orders with Above-Average Sales


In [13]:
# Orders where sales > average sales across all orders
query = """
    SELECT order_id, customer_id, product_id, ROUND(sales, 2) AS sales
    FROM orders
    WHERE sales > (
        SELECT AVG(sales) FROM orders
    )
    ORDER BY sales DESC
    LIMIT 10
"""
result = pd.read_sql(query, conn)
print(f"Orders above average sales (avg = ${pd.read_sql('SELECT ROUND(AVG(sales),2) AS avg FROM orders', conn)['avg'][0]}):")
result


Orders above average sales (avg = $229.86):


,order_id,customer_id,product_id,sales
0,CA-2014-145317,SM-20320,TEC-MA-10002412,22638.48
1,CA-2016-118689,TC-20980,TEC-CO-10004722,17499.95
2,CA-2017-140151,RB-19360,TEC-CO-10004722,13999.96
3,CA-2017-127180,TA-21385,TEC-CO-10004722,11199.97
4,CA-2017-166709,HL-15040,TEC-CO-10004722,10499.97
5,CA-2016-117121,AB-10105,OFF-BI-10000545,9892.74
6,CA-2014-116904,SC-20095,OFF-BI-10001120,9449.95
7,US-2016-107440,BS-11365,TEC-MA-10001047,9099.93
8,CA-2016-158841,SE-20110,TEC-MA-10001127,8749.95
9,CA-2016-143714,CC-12370,TEC-CO-10004722,8399.98


### 4.2 Highest Sales Order per Customer

In [15]:
# For each customer, get the order with their highest single-order sales
query = """
    SELECT o.customer_id, c.customer_name, o.order_id, ROUND(o.sales, 2) AS max_order_sales
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
    WHERE o.sales = (
        SELECT MAX(o2.sales)
        FROM orders o2
        WHERE o2.customer_id = o.customer_id
    )
    ORDER BY max_order_sales DESC
    LIMIT 10
"""
pd.read_sql(query, conn)


,customer_id,customer_name,order_id,max_order_sales
0,SM-20320,Sean Miller,CA-2014-145317,22638.48
1,SM-20320,Sean Miller,CA-2014-145317,22638.48
2,SM-20320,Sean Miller,CA-2014-145317,22638.48
3,SM-20320,Sean Miller,CA-2014-145317,22638.48
4,SM-20320,Sean Miller,CA-2014-145317,22638.48
5,TC-20980,Tamara Chand,CA-2016-118689,17499.95
6,TC-20980,Tamara Chand,CA-2016-118689,17499.95
7,TC-20980,Tamara Chand,CA-2016-118689,17499.95
8,TC-20980,Tamara Chand,CA-2016-118689,17499.95
9,TC-20980,Tamara Chand,CA-2016-118689,17499.95


### 4.3 Total sales per customer 

In [20]:
query = """
    WITH customer_sales AS (
        SELECT
            o.customer_id,
            c.customer_name,
            c.segment,
            c.region,
            ROUND(SUM(o.sales), 2)   AS total_sales,
            ROUND(SUM(o.profit), 2)  AS total_profit,
            COUNT(DISTINCT o.order_id) AS total_orders
        FROM orders o
        JOIN customers c ON o.customer_id = c.customer_id
        GROUP BY o.customer_id, c.customer_name, c.segment, c.region
    )
    SELECT *
    FROM customer_sales
    ORDER BY total_sales DESC
    LIMIT 10
"""
pd.read_sql(query, conn)

,customer_id,customer_name,segment,region,total_sales,total_profit,total_orders
0,SE-20110,Sanjit Engle,Consumer,West,85466.07,18554.74,11
1,AB-10105,Adrian Barton,Consumer,Central,72367.86,27224.03,10
2,KL-16645,Ken Lonsdale,Consumer,West,70876.15,4034.28,12
3,SV-20365,Seth Vernon,Consumer,East,68825.70,7196.55,10
4,SC-20095,Sanjit Chand,Consumer,West,56569.34,23029.65,9
5,SM-20320,Sean Miller,Home Office,South,50086.10,-3961.48,5
6,CJ-12010,Caroline Jumper,Consumer,East,44659.90,3434.97,8
7,CL-12565,Clay Ludtke,Consumer,East,43522.18,7735.13,12
8,CL-12565,Clay Ludtke,Consumer,West,43522.18,7735.13,12
9,LA-16780,Laura Armstrong,Corporate,Central,43366.11,10295.60,11


### 4.4 Customers Who Spent More Than Average Total Sales

In [16]:
query = """
    SELECT customer_id, ROUND(total_sales, 2) AS total_sales
    FROM (
        SELECT customer_id, SUM(sales) AS total_sales
        FROM orders
        GROUP BY customer_id
    ) AS customer_totals
    WHERE total_sales > (
        SELECT AVG(total_sales)
        FROM (
            SELECT customer_id, SUM(sales) AS total_sales
            FROM orders
            GROUP BY customer_id
        )
    )
    ORDER BY total_sales DESC
    LIMIT 10
"""
result = pd.read_sql(query, conn)
print(f"Customers above average total spend: {len(result)} shown (top 10)")
result


Customers above average total spend: 10 shown (top 10)


,customer_id,total_sales
0,SM-20320,25043.05
1,TC-20980,19052.22
2,RB-19360,15117.34
3,TA-21385,14595.62
4,AB-10105,14473.57
5,KL-16645,14175.23
6,SC-20095,14142.33
7,HL-15040,12873.30
8,SE-20110,12209.44
9,CC-12370,12129.07


### 4.5 Rank - Rank Customers based on Total Sales

In [21]:
query = """
    WITH customer_sales AS (
        SELECT
            customer_id,
            ROUND(SUM(sales), 2) AS total_sales
        FROM orders
        GROUP BY customer_id
    )
    SELECT
        customer_id,
        total_sales,
        RANK()       OVER (ORDER BY total_sales DESC) AS rank,
        DENSE_RANK() OVER (ORDER BY total_sales DESC) AS dense_rank
    FROM customer_sales
    ORDER BY rank
    LIMIT 15
"""
pd.read_sql(query, conn)


,customer_id,total_sales,rank,dense_rank
0,SM-20320,25043.05,1,1
1,TC-20980,19052.22,2,2
2,RB-19360,15117.34,3,3
3,TA-21385,14595.62,4,4
4,AB-10105,14473.57,5,5
5,KL-16645,14175.23,6,6
6,SC-20095,14142.33,7,7
7,HL-15040,12873.30,8,8
8,SE-20110,12209.44,9,9
9,CC-12370,12129.07,10,10


### 4.6 ROW_NUMBER — Rank Orders by Sales per Customer

In [22]:
query = """
    SELECT
        customer_id,
        order_id,
        ROUND(sales, 2) AS sales,
        ROW_NUMBER() OVER (
            PARTITION BY customer_id
            ORDER BY sales DESC
        ) AS sales_rank_per_customer
    FROM orders
    ORDER BY customer_id, sales_rank_per_customer
    LIMIT 15
"""
pd.read_sql(query, conn)

,customer_id,order_id,sales,sales_rank_per_customer
0,AA-10315,CA-2016-103982,3930.07,1
1,AA-10315,CA-2014-128055,673.57,2
2,AA-10315,CA-2016-103982,431.98,3
3,AA-10315,CA-2017-147039,362.94,4
4,AA-10315,CA-2014-128055,52.98,5
5,AA-10315,CA-2016-103982,41.72,6
6,AA-10315,CA-2015-121391,26.96,7
7,AA-10315,CA-2014-138100,14.94,8
8,AA-10315,CA-2014-138100,14.56,9
9,AA-10315,CA-2017-147039,11.54,10


### 4.7 Top 3 Customers by Total Sales

In [23]:
query = """
    WITH customer_sales AS (
        SELECT o.customer_id, c.customer_name, c.segment,
               ROUND(SUM(o.sales), 2) AS total_sales,
               ROUND(SUM(o.profit), 2) AS total_profit
        FROM orders o
        JOIN customers c ON o.customer_id = c.customer_id
        GROUP BY o.customer_id, c.customer_name, c.segment
    )
    SELECT customer_name, segment, total_sales, total_profit,
           RANK() OVER (ORDER BY total_sales DESC) AS rank
    FROM customer_sales
    LIMIT 3
"""
top_customers = pd.read_sql(query, conn)
top_customers

,customer_name,segment,total_sales,total_profit,rank
0,Ken Lonsdale,Consumer,141752.29,8068.55,1
1,Sanjit Engle,Consumer,134303.82,29157.45,2
2,Adrian Barton,Consumer,130262.14,49003.25,3


## 5. Final Combined Query <a id='subqueries'></a>

### 5.1 This query gives **Customer Name, Total Sales, and their Sales Rank** in one shot.

In [31]:
query = """
WITH customer_sales AS (
    SELECT
        o.customer_id,
        ROUND(SUM(o.sales), 2) AS total_sales
    FROM orders o
    GROUP BY o.customer_id
)

SELECT
    c.customer_name,
    cs.total_sales,
    RANK() OVER (ORDER BY cs.total_sales DESC) AS rank
FROM customer_sales cs
JOIN customers c
    ON cs.customer_id = c.customer_id
ORDER BY rank;
"""
pd.read_sql(query, conn)


,customer_name,total_sales,rank
0,Sean Miller,25043.05,1
1,Sean Miller,25043.05,1
2,Sean Miller,25043.05,1
3,Sean Miller,25043.05,1
4,Sean Miller,25043.05,1
...,...,...,...
4683,Mitch Gastineau,16.74,4684
4684,Carl Jackson,16.52,4685
4685,Lela Donovan,5.30,4686
4686,Thais Sissman,4.83,4687


## MINI PROJECT : Customers Sales Insigts <a id='subqueries'></a>

### 6.1 Top 5 Customers (Top Sales)

In [33]:
query = """
WITH customer_sales AS (
    SELECT
        customer_id,
        ROUND(SUM(sales), 2) AS total_sales
    FROM orders
    GROUP BY customer_id
),
customer_info AS (
    SELECT DISTINCT
        customer_id,
        customer_name
    FROM customers
)

SELECT
    ci.customer_name,
    cs.total_sales,
    RANK() OVER (ORDER BY cs.total_sales DESC) AS rank
FROM customer_sales cs
JOIN customer_info ci
    ON cs.customer_id = ci.customer_id
ORDER BY rank
LIMIT 5;
"""
pd.read_sql(query, conn)


,customer_name,total_sales,rank
0,Sean Miller,25043.05,1
1,Tamara Chand,19052.22,2
2,Raymond Buch,15117.34,3
3,Tom Ashbrook,14595.62,4
4,Adrian Barton,14473.57,5


### 6.2 Bottom 5 Customers (Lowest Sales)

In [34]:
query = """
    WITH customer_sales AS (
        SELECT o.customer_id, c.customer_name, c.segment,
               ROUND(SUM(o.sales), 2) AS total_sales
        FROM orders o
        JOIN customers c ON o.customer_id = c.customer_id
        GROUP BY o.customer_id, c.customer_name, c.segment
    )
    SELECT customer_name, segment, total_sales,
           RANK() OVER (ORDER BY total_sales ASC) AS low_rank
    FROM customer_sales
    LIMIT 5
"""
pd.read_sql(query, conn)


,customer_name,segment,total_sales,low_rank
0,Lela Donovan,Corporate,5.30,1
1,Thais Sissman,Consumer,9.67,2
2,Carl Jackson,Corporate,16.52,3
3,Mitch Gastineau,Corporate,16.74,4
4,Roy Skaria,Home Office,44.66,5


### 6.3 Customers with Only One Order 

In [40]:
query = """
WITH customer_orders AS (
    SELECT
        customer_id,
        COUNT(DISTINCT order_id) AS total_orders
    FROM orders
    GROUP BY customer_id
),
customer_info AS (
    SELECT DISTINCT
        customer_id,
        customer_name
    FROM customers
)

SELECT
    ci.customer_name,
    co.total_orders
FROM customer_orders co
JOIN customer_info ci
    ON co.customer_id = ci.customer_id
WHERE co.total_orders = 1
ORDER BY ci.customer_name;
"""
pd.read_sql(query, conn)


,customer_name,total_orders
0,Anemone Ratner,1
1,Anthony O'Donnell,1
2,Carl Jackson,1
3,Jenna Caffey,1
4,Jocasta Rupert,1
5,Lela Donovan,1
6,Mitch Gastineau,1
7,Patricia Hirasaki,1
8,Ricardo Emerson,1
9,Roland Murray,1


### 6.4 Customers with Above-Average Sales 

In [42]:
query = """
WITH customer_sales AS (
    SELECT
        customer_id,
        ROUND(SUM(sales), 2) AS total_sales
    FROM orders
    GROUP BY customer_id
)

SELECT
    c.customer_name,
    cs.total_sales
FROM customer_sales cs
JOIN (
    SELECT DISTINCT customer_id, customer_name
    FROM customers
) c
ON cs.customer_id = c.customer_id
WHERE cs.total_sales > (
    SELECT AVG(total_sales)
    FROM customer_sales
)
ORDER BY cs.total_sales DESC;
"""
pd.read_sql(query, conn)


,customer_name,total_sales
0,Sean Miller,25043.05
1,Tamara Chand,19052.22
2,Raymond Buch,15117.34
3,Tom Ashbrook,14595.62
4,Adrian Barton,14473.57
...,...,...
289,Julie Kriz,2932.48
290,Shaun Weien,2921.54
291,Maris LaWare,2921.50
292,Rob Dowd,2912.89


### 6.5 Highest order value per customer

In [43]:
query = """
WITH customer_max_order AS (
    SELECT
        customer_id,
        ROUND(MAX(sales), 2) AS highest_order_value
    FROM orders
    GROUP BY customer_id
)

SELECT
    c.customer_name,
    cmo.highest_order_value
FROM customer_max_order cmo
JOIN (
    SELECT DISTINCT
        customer_id,
        customer_name
    FROM customers
) c
ON cmo.customer_id = c.customer_id
ORDER BY cmo.highest_order_value DESC;
"""
pd.read_sql(query, conn)

,customer_name,highest_order_value
0,Sean Miller,22638.48
1,Tamara Chand,17499.95
2,Raymond Buch,13999.96
3,Tom Ashbrook,11199.97
4,Hunter Lopez,10499.97
...,...,...
788,Carl Jackson,16.52
789,Mitch Gastineau,12.32
790,Roy Skaria,9.65
791,Lela Donovan,5.30


## 7. Summary & Insights <a id='summary'></a>

### Key SQL Techniques Used

| Technique | Purpose | Used In |
|---|---|---|
| **Subquery** | Filter rows based on aggregated values | Above-avg sales, max order per customer |
| **CTE** | Reusable named result sets for cleaner queries | Customer totals, category summary |
| **ROW_NUMBER()** | Assign unique rank within a partition | Orders ranked per customer |
| **RANK() / DENSE_RANK()** | Rank with gap / without gap on ties | Customer sales ranking |
| **SUM() OVER (...)** | Running totals across ordered rows | Cumulative daily sales |
| **JOIN** | Combine normalized tables | All business queries |

---

### 📊 Business Insights

1. **Revenue Depends Heavily on a Few Customers** — A limited number of customers account for a large share of overall sales. Building relationships with these high-value customers can help maintain consistent revenue growth.

2. **Customer Retention Needs Attention** — Many customers appear to purchase only once and do not return. Introducing loyalty programs or personalized follow-ups could encourage repeat purchases.

3. **Larger Discounts Do Not Guarantee Better Returns** — Products receiving higher discounts often generate lower profits. The company should review its pricing and discount policies to balance sales growth with profitability.

4. **High-Value Orders Are Concentrated in Specific Categories** — Orders with above-average sales are mainly associated with categories such as Technology and Furniture.

5. **Performance Differs Across Regions** — Sales and profit levels vary significantly between regions. Understanding the factors behind strong-performing regions can help improve results in weaker markets.

